# Delay Locked Loop Explainer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SiliconJackets/Cac_Spring26/blob/main/DelayLockedLoop.ipynb)

```
Copyright 2026 SiliconJackets @ Georgia Institute of Technology
SPDX-License-Identifier: Apache-2.0
```

|Name|Affiliation| Email |IEEE Member|SSCS Member|
|:--:|:----------:|:----------:|:----------:|:----------:|
|Ethan Huang|Georgia Institute of Technology|ethanhuang@gatech.edu|No|No|
|Zheng-Yin Lee|Georgia Institute of Technology|zlee63@gatech.edu|No|No|
|Alfi Misha Antony Selvin Raj|Georgia Institute of Technology|alfiselvin@gatech.edu|No|No|
|Oliver S. Lee|Georgia Institute of Technology|xli3086@gatech.edu|Yes|No|
|Shreyas Angadi|Georgia Institute of Technology|sangadi6@gatech.edu|No|No|
|Mythri Muralikannan|Georgia Institute of Technology|mmuralikannan3@gatech.edu|Yes|Yes|

This notebook provides an interactive exploration of a Delay Locked Loop (DLL), illustrating the functionality and design variations of key submodules: the Phase Detector, Controller, DCDL, and Ring Oscillator. It analyzes the behavior and trade-off associated with each component with SPICE simulations. The notebook culminates in a design that measures the delay between two arbitrary clock signals, with the appropriate submodules are chosen through our preceding analysis. The design targets the [open source sky130 PDK](https://github.com/google/skywater-pdk/) by Google and Skywater.

In [46]:
# @title Download Dependencies {display-mode: "form"}
# @markdown
# @markdown Click the ▷ button to download all neccesary dependencies (librelane + nix-os)
# @markdown as well as all of our RTL files + scripts for this interactive notebook
# @markdown
import os
from pathlib import Path
import subprocess
import sys
import shutil
import tempfile

# NgSpice Installation
!sudo apt-get update
!sudo apt-get install -y ngspice

# Nix Installation
os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            installer_path = d / "nix"
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            with subprocess.Popen(
                [
                    "bash",
                    installer_path,
                    "install",
                    "--prefer-upstream-nix",
                    "--no-confirm",
                    "--extra-conf",
                    "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                encoding="utf8",
            ) as p:
                for line in p.stdout:
                    print(line, end="")
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"


# Librelane Installation
import os
import yaml
import subprocess
import IPython

librelane_version = "latest"  # @param {key:"LibreLane Version", type:"string"}

if librelane_version == "latest":
    librelane_version = "main"

pdk_root = "~/.ciel"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

librelane_ipynb_path = os.path.join(os.getcwd(), "librelane_ipynb")

display(IPython.display.HTML("<h3>Downloading LibreLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {librelane_ipynb_path}
!mkdir -p {librelane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {librelane_ipynb_path}
else:
    !curl -L "https://github.com/librelane/librelane/tarball/{librelane_version}" | tar -xzC {librelane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading LibreLane's dependencies…</a>"))
try:
    with subprocess.Popen(
        [
            "nix",
            "profile",
            "install",
            ".#colab-env",
        ],
        cwd=librelane_ipynb_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf8",
    ) as p:
        for line in p.stdout:
            print(line, end="")
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=librelane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import ciel
from ciel.source import StaticWebDataSource

with open(
    os.path.join(librelane_ipynb_path, "librelane", "pdk_hashes.yaml"), "r"
) as file:
    pdk_hashes = yaml.safe_load(file)

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    pdk_hashes[pdk],
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)

sys.path.insert(0, librelane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

#Install our scripts
%cd /content/
!rm -rf CAC_2026
!git clone https://github.com/SiliconJackets/CaC_Spring26 CAC_2026

#Install iverilog
!sudo apt-get install -y iverilog

#Install VCD reader
!pip install vcdvcd -q

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Fetched 388 kB in 2s (219 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ngspice is already the ne

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 8900k    0 8900k    0     0  6953k      0 --:--:--  0:00:01 --:--:-- 46.3M


Version 8afc8346a57fe1ab7934ba5a6056ea8b43078e71 enabled for the sky130 PDK.

/content
Cloning into 'CAC_2026'...
remote: Enumerating objects: 1878, done.
remote: Counting objects: 100% (366/366), done.
remote: Compressing objects: 100% (236/236), done.
remote: Total 1878 (delta 192), reused 274 (delta 123), pack-reused 1512 (from 1)
Receiving objects: 100% (1878/1878), 16.20 MiB | 11.91 MiB/s, done.
Resolving deltas: 100% (537/537), done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove and 24 not upgraded.


## What is a Delay-Locked Loop
---

A Delay-locked Loop (DLL) is a closed-loop feedback circuit that aligns the phase of an output clock to a reference clock. It shifts the phase of this output clock by delaying it, either through a chain of discrete digital delay cells or through a voltage-controlled analog delay line, until the output edge lines up with the reference edge. Unlike a PLL, a DLL does not generate a new frequency.

The DLL's core feedback loop consists of three blocks: a Phase Detector, a Controller, and a delay line, for which this notebook uses a Digitally Controlled Delay Line (DCDL). Together, they form a feedback loop that continuously adjusts the delay applied to the output clock until its phase matches the reference, at which point, the system is "locked."

A Ring Oscillator is also implemented to provide a tunable clock source for testing.

> **TODO: Add draw.io diagram (top-level DLL block diagram: CLK_REF -> Phase Detector -> Controller -> DCDL -> CLK_OUT feedback loop)**

Each module section below describes the high-level function, presents each implementation variant with its pros and cons, as evidenced by the spice simulations.

## Phase Detector
---

The phase detector compares the rising edges of the reference clock (`clk_in`) and the feedback clock (`clk_out`) and produces two single-bit outputs: `up` (reference leads, increase the delay) and `down` (feedback leads, decrease the delay). It is important to note that the phase detector does not measure the magnitude of the phase error, only the direction.

A bang-bang style of detection is simple, but it means the loop can never settle perfectly still. Near lock, the detector continuously toggles between `up` and `down`, producing jitter. Other phase detector architectures, as explored below, can avoid this intrinsic toggling, but at the cost of added complexity. The choice of phase detector architecture directly impacts the loop's accuracy, stability, and susceptibility to metastability.

> **TODO: Add waveform GIF (phase_detector.gif) showing clk_in, clk_out, up, down signals during lock acquisition and steady state**

> **TODO: Add waveform explanation describing how up/down toggle during acquisition and settle into a limit cycle near lock**

---
### Implementation 1: Single Flip-Flop Detector

> **TODO: Add draw.io diagram (single D flip-flop: clk_out on D, clk_in on CLK, Q driving up/down decode)**

This is perhaps the most trivial phase detector. A single D type flip-flop samples `clk_out` on the rising edge of `clk_in`. If `clk_out` is low at the sampling instant, the feedback clock is trailing behind the reference clock, hence `up` is asserted. Conversely, `down` is asserted if `clk_out` is high at the sampling instant.

There is no "equal" state; the detector always picks a direction.

| Pros | Cons |
|----------|----------|
| Extremely low area and complexity. A single flip-flop plus output decode. Smallest power and area footprint of all detectors. | Inherently biased. When clocks are perfectly aligned, the detector still asserts `down` because `clk_out` is sampled as high at the rising edge of `clk_in`. The bias means the loop locks with a small systematic phase offset rather than true zero error. |
| Fully synthesizable, fast timing. Minimal critical path yields high maximum operating frequency (best performance). | No frequency detection capability. |
| | Persistent bang-bang toggling near lock limits achievable jitter performance. |


In [ ]:
!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_ff1

-> Starting flow for 'phase_detector_syn_ff1'...
/content/CAC_2026/librelane/design/phase_detector_syn_ff1/config.json
──────────────────────────────── Verilator Lint ────────────────────────────────
────────────────────────── Lint Timing Errors Checker ──────────────────────────
───────────────────────────── Lint Errors Checker ──────────────────────────────
──────────────────────────── Lint Warnings Checker ─────────────────────────────
───────────────────────────── Generate JSON Header ─────────────────────────────
────────────────────────────────── Synthesis ───────────────────────────────────
──────────────────────────── Unmapped Cells Checker ────────────────────────────
────────────────────────────── Yosys Synth Checks ──────────────────────────────
─────────────────────── Netlist Assign Statement Checker ───────────────────────
─────────────────────────────── Check SDC Files ────────────────────────────────
──────────────────────────── Check Macro Instances ────────────────────


> **TODO: Add simulation results / waveform figures for single-FF detector**

> **TODO: Add data analysis (bias characterization, jitter measurements)**

---

### Implementation 2: Edge-Order Detector

> **TODO: Add draw.io diagram (two cross-coupled latches driven by clk_in and clk_out, outputs up/down)**

This is an extension of the single flip-flop detector. Each clock edge tries to assert its corresponding output (`up` or `down`), but only succeeds if the other clock's edge has not already arrived. When `clk_in` rises first, `up` is set; when `clk_out` rises first, `down` is set. The opposite edge clears the output, completing the comparison for that cycle.

| Pros | Cons |
|----------|----------|
| Minimal logic. Two flip-flop-style blocks with cross-coupled gating. Low area and power, only marginally more than the single-FF detector. | Race conditions when edges arrive nearly simultaneously; both `up` and `down` can briefly assert (observed in aligned-edge test cases). |
| No explicit reset feedback path needed. | When `clk_out` leads `clk_in`, the detector can still incorrectly assert `up` because the first-edge-wins logic lets `clk_in` capture before `down` registers the earlier feedback edge. |
| | No frequency detection capability. Performance is limited by metastability susceptibility when edges are close. |


In [ ]:
!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_edge


> **TODO: Add simulation results / waveform figures for edge-order detector**

> **TODO: Add data analysis (measured accuracy across different phase offsets)**

---



### Implementation 3: Phase-Frequency Detector (PFD)

> **TODO: Add draw.io diagram (two FFs with D=1 clocked by clk_in/clk_out, AND gate feeding async reset to both)**

This is the industry-standard architecture. The analysis below will prove why. Two flip-flops are held with D=1. The rising edge of `clk_in` sets the first FF (asserting `up`), and the rising edge of `clk_out` sets the second FF (asserting `down`). An AND gate generates a reset pulse that clears both flip-flops when both outputs are simultaneously high. The width of the `up` or `down` pulse is proportional to the phase difference between the two clocks. This implementation can also detect frequency mismatches; if one clock is faster, its corresponding output will be asserted for longer durations, on average, driving the loop in the correct direction. It is important to note that this implementation is well suited for PLLs, the frequency detection capability is unnecessary in a DLL. Only the phase-detection behavior is used.

| Pros | Cons |
|----------|----------|
| Detects both phase error and frequency error. The only detector here with frequency acquisition capability. | Dead zone. When phase difference is very small, the reset pulse fires almost immediately, producing narrow output pulses that may not be captured reliably by downstream logic. |
| Pulse width encodes error magnitude (useful for analog loops, and provides clean direction for digital bang-bang use). | Slightly more area and power than single-FF or edge-order approaches (two flip-flops, AND gate, and reset feedback path). |
| Well-understood, widely used in PLL/DLL designs. Robust and predictable behavior across PVT corners. | The AND-gate reset path introduces a minimum pulse width constraint that can limit maximum operating frequency. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_pfd



> **TODO: Add simulation results / waveform figures for PFD**

> **TODO: Add data analysis (dead zone characterization, frequency acquisition behavior)**

---




### Implementation 4: Sampled XOR Detector

> **TODO: Add draw.io diagram (XOR gate with clk_in/clk_out inputs, output sampled by DFF clocked on clk_in, decode to up/down)**

An XOR gate compares the two clocks. If the signals mismatch, `1` will be produced. This signal is then sampled on the rising edge of teh reference clock to determine direction. If the feedback clock is low at the sampling instant, the reference is leading (`up`); if the feedback clock is high, the feedback is leading (`down`); if b oth clocks agree, both outputs are cleared.

However, a fundamental problem with this architecture is that the XOR gate and the direction sample are evaluated at the same instant, the reference clock's rising edge. Reference clock is always high at this moment. The XOR reduces to simply checking whether the feedback clock is low or high. When the feedback clock is low, the XOR gate sees a mismatch and `up` is asserted. When the feedback clock is high, the XOR gate sees no mismatch and both outputs are cleared. The `down` path is never reached. **The detector can only increase delay or hold; it cannot decrease delay.** This makes the loop fundamentally one-directional.

| Pros | Cons |
|----------|----------|
| Simple combinational front-end (single XOR gate). Minimal area overhead for the detection logic. | One-directional: the detector can only assert `up` or clear both outputs. It cannot assert `down`, so the DLL loop can only increase delay, never decrease it. |
| Naturally produces a "no error" state when clocks are aligned (both outputs cleared). | XOR detects mismatch but not direction; direction is inferred by sampling the feedback clock level, making it duty-cycle and timing dependent. |
| Low power due to minimal switching activity (one XOR gate, one flip-flop). Smallest dynamic power of all detectors. | Not suitable as a standalone detector in practical designs due to the single-sided output limitation. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_xor1

In [45]:
# @markdown
# @markdown Click the ▷ button to graph the functional waveform of how the XOR Phase Detector should work
# @markdown

design_file = "/content/CAC_2026/design_root/src/verilog/parts/phase_detector/phase_detector_syn_xor1.sv"
testbench_file = "/content/CAC_2026/design_root/src/verilog/parts/phase_detector/tb_phase_detector_syn_xor1.sv"

!iverilog -o simulation.vvp {design_file} {testbench_file}

!vvp simulation.vvp

import sys
import os
from pathlib import Path

# Define paths and resolve '~' to an absolute path
PROJECT_ROOT = Path("~/CaC_Spring26").expanduser()
SPICE_DIR = PROJECT_ROOT / "spice"
scripts_path = str(PROJECT_ROOT / "scripts")

# Add the scripts folder to the system path
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

from analysis_util import find_switching_time, get_sample, apply_time_shift
from plot_framework import iplot, ioverlay, isweep, istack, isweep_overlay
from ngspice_loader import load_wrdata, load_sweep
from vcdvcd import VCDVCD

from bokeh.io import output_notebook
output_notebook()  # Initialize Bokeh to display inline in the Colab notebook

vcd_file_path = '/content/tb_phase_detector_syn_xor1.vcd'

if not os.path.exists(vcd_file_path):
    print(f"Error: Could not find '{vcd_file_path}'. Make sure it is uploaded to Colab.")
else:
    vcd = VCDVCD(vcd_file_path)

    end_time = 0
    for sig in vcd.signals:
        if vcd[sig].tv:
            last_event_time = vcd[sig].tv[-1][0]
            if last_event_time > end_time:
                end_time = last_event_time

    def get_vcd_trace(sig_name):
        if sig_name not in vcd.signals:
            print(f"Warning: {sig_name} not found in VCD.")
            return ([0, end_time], [0, 0])

        signal_data = vcd[sig_name]
        times = []
        values = []
        for time, val_str in signal_data.tv:
            times.append(time)

            if val_str.lower() in ('x', 'z'):
                val = 0.5
            elif val_str.startswith('b'):
                try: val = int(val_str[1:], 2)
                except ValueError: val = 0
            else:
                try: val = int(val_str)
                except ValueError: val = 0
            values.append(val)

        if times:
            times.append(end_time)
            values.append(values[-1])
        else:
            times = [0, end_time]
            values = [0, 0]

        return (times, values)

    prefix = "tb_phase_detector.dut."
    traces = {
        "CLK_IN": get_vcd_trace(f"{prefix}clk_in"),
        "CLK_OUT": get_vcd_trace(f"{prefix}clk_out"),
        "RST": get_vcd_trace(f"{prefix}rst"),
        "PHASE_ERROR": get_vcd_trace(f"{prefix}phase_error"),
        "UP": get_vcd_trace(f"{prefix}up"),
        "DOWN": get_vcd_trace(f"{prefix}down"),
    }

    istack(
        layers=[
            {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
            {"RST": traces["RST"]},
            {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        ],
        title="Verilog FF1 Phase Detector — clk_out leads",
        xlabel="Time (ps)",   # Based on standard `timescale 1ps/1ps
        ylabels=["Clocks (Logic)", "Reset", "Phase Err", "Outputs"],
        kind="step",          # Forces square digital waveforms
        layer_height=150,     # Adjusted height for 4 layers
    )

> **TODO: Add simulation results / waveform figures for XOR detector**

> **TODO: Add data analysis (duty cycle sensitivity, directional bias measurements)**

## Controller
---

The controller recieves the `up`/`down` signals from the phase detector and maintains an N-bit control word that sets the delay through the DCDL. At a high level, every controller variant is an accumulator. When `up` is asserted, the control word increments (more delay), when `down` is asserted, the control world decrements (less delay), and when both match, it holds. The control word is clamped to `[0, 2^N - 1]` to prevent wrap-around.

The differentionation between the implementions below is how aggressively they update. The step size, filtering, and mode-switching strategies each trade off lock speed against steady-state jitter. The phase detector provides direction, the controller sets the "bandwidth". Large steps mean fast convergence, but more overshoot. Small steps more smooth tracking, but slower acquisition.

> **TODO: Add waveform GIF (controller.gif) showing up, down, ctrl[N-1:0] evolving over time during lock acquisition and steady state**

> **TODO: Add waveform explanation describing ctrl ramping during acquisition, settling into limit-cycle toggling near lock**

---

### Implementation 1: Saturating Controller

> **TODO: Add draw.io diagram (adder/subtractor with up/down inputs, saturation clamp, ctrl output register)**

This is the trivial, baseline design. The implementation is the high level overview mentioned above. On each clock edge, the control world increments or decrements by 1 depending on if `up` or `down` is asserted. Hard saturation clamps `ctrl` at 0 and `2^N - 1` to prevent underflow or overflow. On reset, the control world is initialized to a parameterized `INIT_CTRL` value (default midpoint of 32 for 6-bit control).

| Pros | Cons |
|----------|----------|
| Simplest possible controller. Easy to verify and reason about. | Fixed +-1 step means slow convergence when far from lock. |
| Predictable, monotonic response to persistent error. | Near lock, the +-1 toggling produces a limit cycle with 1-LSB jitter that cannot be reduced without external filtering. |
| Smallest area and lowest power of all controllers: one adder/subtractor, one comparator, one register. Industry baseline for comparison. | No adaptability. Performance is the same whether the loop is far from or near lock. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py TODO



> **TODO: Add simulation results / waveform figures for saturating controller**

> **TODO: Add data analysis (lock time, steady-state jitter amplitude)**

---

### Implementation 2: Filtered Controller

> **TODO: Add draw.io diagram (up/down counters feeding threshold comparators, gating +-1 update to ctrl register)**

Instead of updating the control word on every cycle, this controller requires `FILTER_LEN` consecutive `up` (or `down`) requests before incrementing or decrementing by 1. Two internal counters track consecutive `up` and `down` assertions. If the direction changes or both signals are idle, both counters reset to zero. Only when one counter reaches the threshold does the control word update, and then that counter resets. This acts as a simple digital low-pass filter on the phase detector output.

| Pros | Cons |
|----------|----------|
| Significantly reduces jitter near lock; transient noise or single-cycle glitches from the phase detector are ignored. | Slower lock acquisition. The loop must see `FILTER_LEN` consistent cycles before taking any action. |
| Steady-state limit cycle is suppressed because the toggling `up`/`down` pattern resets the counters before threshold is reached. | The `FILTER_LEN` parameter requires tuning: too small and it behaves like the saturating controller, too large and the loop becomes unresponsive. |
| Modest area increase over the saturating controller (two additional counters and comparators). Power consumption is slightly higher due to counter activity, but reduced output toggling saves downstream switching power. | Effective loop bandwidth is reduced by the filter factor, which can be problematic if the clock environment changes rapidly. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py TODO



> **TODO: Add simulation results / waveform figures for filtered controller**

> **TODO: Add data analysis (lock time vs FILTER_LEN, jitter reduction compared to saturating)**

---

### Implementation 3: Acquire/Track Controller

> **TODO: Add draw.io diagram (mode FSM selecting between large/small step size, quiet counter triggering mode switch, adder/subtractor with saturation)**

The acquire/track controller operates in two modes to balance a fast lock acquisition against low steady-state jitter. On startup, it is in aquire mode and adjusts the delay by a coarse step (`ACQUIRE_STEP`, default 4) on every `up`/`down` assertion from the phase detector. This allows the DLL to converge rapidly towards the lock point. A quiet-cycle counter monitors phase-detector activity. Each cycle without an `up` or `down` assertion increments a counter, and any new assertion clears it. When this counter reaches the threshold `QUIET_CYCLES` (default 8), the loop is considered near lock and the controller goes into track mode. Here, the corrections uses a small step size (`TRACK_STEP`, default 1) to minimize jitter. The mode transition is one-way and held until reset, so transient disturbances after lock cannot kick the loop back into coarse-step behavior.

| Pros | Cons |
|----------|----------|
| Fast acquisition (large steps cover the control range quickly) with low steady-state jitter (small steps near lock). | One-way mode switch means the controller cannot re-acquire if the loop is disturbed after locking (e.g., supply noise kicks the delay far from lock). |
| Widely used in industry DLL/PLL designs as a practical balance of speed and stability. | The `QUIET_CYCLES` threshold must be tuned; too aggressive and the controller switches to track mode before actually reaching lock. |
| Moderate area and power - adds only a mode register, quiet counter, and step-size mux over the saturating baseline. | Larger step sizes during acquire mode can cause overshoot, especially if `ACQUIRE_STEP` is too large relative to the DCDL resolution. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py TODO



> **TODO: Add simulation results / waveform figures for acquire/track controller**

> **TODO: Add data analysis (acquisition phase duration, jitter comparison between modes)**

---

### Implementation 4: Coarse/Fine Controller

> **TODO: Add draw.io diagram (two separate counters for coarse/fine with mode select, concatenated into ctrl output)**

The controller partitions its control word into two concatenated fields `ctrl = {coarse, fine}`. On startup it opereates in course mode, where each `up`/`down` assertion increments or decrements only the course field. This allows the DLL to converge rapidly towards the lock point. A quiet-cycle counter monitors phase-detector activity. When this counter reaches the threshold `SWITCH_QUIET` (default 8), the loop is considered near lock and the controller transitions to fine mode. Subsequence corrections adjust only the fine field. Each field saturates independently at its own bounds, preventing carries between fields and cleanly decoupling the two adjustment ranges.

| Pros | Cons |
|----------|----------|
| High total resolution without needing an extremely wide control word; coarse bits provide range, fine bits provide precision. | More complex than a single-counter design - two independent saturating counters plus mode logic. Larger area than the saturating or filtered controllers. |
| Clean separation of acquisition (coarse) and tracking (fine) phases. | Coarse-to-fine boundary can introduce a jump if the coarse step doesn't land near the correct fine range. |
| Hardware-efficient: the two counters are small and independent. Power overhead is modest since only one counter is active at a time. | Same one-way mode switch limitation as acquire/track - cannot re-acquire after locking. |

It can be noted that acquire/track and coarse/fine are two implementations of the same dual-speed control strategy. It only differs in whether the two speeds come from changing the step size or from changing which bits of the control word are being adjusted.

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py controller_2mode



> **TODO: Add simulation results / waveform figures for coarse/fine controller**

> **TODO: Add data analysis (resolution improvement, coarse-to-fine transition behavior)**

---

### Implementation 5: Variable-Step Controller

> **TODO: Add draw.io diagram (direction tracker with same_dir_count, step-size lookup table, adder/subtractor with saturation)**

This controller adapts its step size based on the consecutive `up`/`down` assertions. There exists a counter, and each repeat increments the counter, while any direction reversal resets it to 1. The step size is then selected from the counter value in three tiers, a unit step of 1 by default, growing to `MED_STEP` (default 2) once the count exceeds `MED_THRESH` (default 4), and jumping again to BIG_STEP (default 4) once it exceeds `BIG_THRESH` (default 8). Persistent same-direction error is interpreted the loop is far from the lock point, so the controller "sprints" towards it with progressively larger corrections. As soon as the phase detector begins alternating firections, the counter collapses and the loop reverts to fine adjustments.

| Pros | Cons |
|----------|----------|
| Fastest convergence of all five controllers - accelerates when far from lock without requiring an explicit mode switch. | More tuning parameters (`MED_THRESH`, `BIG_THRESH`, `MED_STEP`, `BIG_STEP`); poor tuning can cause overshoot or oscillation. |
| Naturally adapts to both acquisition and tracking without separate modes. Can re-accelerate if disturbed, unlike the one-way mode controllers. | Near lock, even a short run of consistent `up` or `down` from the limit cycle can trigger medium steps, potentially increasing jitter compared to a fixed +-1 controller. |
| Moderate area cost - adds a direction register, 4-bit counter, and step-size comparators/mux. Power scales with activity since larger steps mean fewer total update cycles to reach lock. | Most complex controller to verify. Non-linear step behavior makes worst-case jitter harder to bound analytically. |


In [ ]:
!python3 CAC_2026/scripts/flow_cli.py controller_variable_step


> **TODO: Add simulation results / waveform figures for variable-step controller**

> **TODO: Add data analysis (convergence speed comparison, overshoot characterization)**

## DCDL (Digitally Controlled Delay Line)
---

The DCDL is the actuator of the DLL. It receives the digital control word from the controller and converts it into a physical propagation delay applied to the clock signal. The clock enters a chain of delay elements, and the control word determines how many stages the signal passes through before reaching the output. A larger control value means more active delay stages and a longer delay.

The DCDL must provide monotonic, glitch-free delay adjustment across its full control range. What varies between the implementations below is the delay primitive (inverters vs. NAND gates), the tap selection strategy (mux trees vs. bypass logic), and how each deals with signal polarity. They span from simple inverter chains to dual-chain vernier interpolation.

---

### Implementation 1: Inverter-Based DCDL

> **TODO: Add draw.io diagram (4-stage inverter chain with pairs of inverters per stage, hierarchical 2-level mux tree, output Y)**

> **TODO: Generate waveform GIF for inverter-based DCDL with correct timings**

Each delay stage consists of two inverters in series, which preserves signal polarity (an even number of inversions). The input propagates through the chain, producing one tap per stage. A hierarchical mux tree, controlled by the control word, selects which tap is routed to the output. This implementation uses a chain of 4 double-inverter stages with a 2-bit control word.

| Pros | Cons |
|----------|----------|
| Straightforward design; the double-inverter stages guarantee consistent polarity at every tap. | Two inverters per stage means more area and power per delay step compared to single-inverter designs. |
| Hierarchical mux tree scales cleanly with more stages. | Mux tree adds its own propagation delay to every path, which may limit minimum achievable delay. |
| Easy to reason about delay linearity. Good baseline for area vs. resolution comparison. | Coarse resolution. Extending to more taps requires a wider chain and deeper mux tree, increasing area and power. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py inv_dcdl



> **TODO: Add simulation results / waveform figures for inverter-based DCDL**

> **TODO: Add data analysis (delay per step, linearity measurement)**

---

### Implementation 2: Conditional Inverter DCDL

> **TODO: Add draw.io diagram (single-inverter chain with 4 stages, mux tree, XNOR gate at output for polarity correction)**

> **TODO: Generate waveform GIF for conditional inverter DCDL with correct timings**

This design uses a single inverter per stage instead of two, halving the chain length for the same number of taps. The trade-off is that odd-numbered taps produce an inverted signal (odd number of inversions from the input), while even-numbered taps are non-inverted. A mux tree selects the desired tap, and an XNOR gate at the output conditionally flips the polarity based on a control bit.

However, the polarity correction is imperfect in this implementation. The XNOR keys on one bit of the control word, but the actual inversion parity of the selected tap depends on a different bit. The correction only produces the correct output polarity for half of the control word values; for the other half, the output is inverted.

| Pros | Cons |
|----------|----------|
| Half the inverters of the double-inverter design for the same number of taps. Significantly smaller area and lower power. | The XNOR correction gate adds a fixed delay to every output path, and its own delay variation can affect linearity. |
| Finer delay granularity per stage (each inverter adds one gate delay instead of two). | Polarity correction is only valid for half the control word values. The remaining values produce an inverted output. |
| Lower power consumption due to fewer switching elements. | Extending to wider control words requires careful bookkeeping to maintain correct polarity across all tap selections. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py inv_dcdl_cond


> **TODO: Add simulation results / additional waveform figures for conditional inverter DCDL**

> **TODO: Add data analysis (delay per step vs. Implementation 1, area comparison)**

---

### Implementation 3: Glitch-Free Inverter DCDL

> **TODO: Add draw.io diagram (3-stage inverter chain, one-hot decoder, registered sel_reg, NAND2 gating per tap, NAND tree to output Y)**

> **TODO: Generate waveform GIF for glitch-free DCDL with correct timings**

This variant addresses glitch hazards that occur in the mux-based designs when the select signal changes while the clock is propagating through the chain. The approach replaces the mux tree with a NAND-based gating and combining scheme. The control word is decoded into a one-hot signal and registered on the clock edge, so the active tap can only change synchronously. Each tap is inverted and gated with its one-hot select bit through a NAND gate, and the gated outputs are combined through a NAND tree to produce the final output.

The first tap is wired directly to the input (zero inverter delay), and each subsequent tap adds one inverter delay. On reset, the registered select defaults to the first tap.

| Pros | Cons |
|----------|----------|
| Glitch-free by construction; the registered one-hot select ensures no transient paths are created during control word changes. | Requires a clock and reset input, unlike the purely combinational implementations above. Adds sequential area for the registered select. |
| NAND-based gating and combining is area-efficient in standard cell libraries. | The registered select adds one cycle of latency between a control word change and the delay actually updating. This impacts loop response time. |
| Eliminates potential timing violations caused by mid-transition glitches in the other DCDL variants. | The NAND tree path adds fixed delay overhead. More total gates than the simple mux-tree approaches, increasing area and static power. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py TODO



> **TODO: Add simulation results / additional waveform figures for glitch-free DCDL**

> **TODO: Add data analysis (glitch characterization vs. Implementations 1 and 2)**

---

### Implementation 4: NAND-Based DCDL

> **TODO: Add draw.io diagram (4-stage chain of nand_dcdl_cells, each cell with in0/in1/ctl inputs showing bypass/propagate logic)**

> **TODO: Generate waveform GIF for NAND-based DCDL with correct timings**

A fundamentally different approach. Instead of selecting a tap from a shared delay chain, each stage is a NAND-based cell that can either propagate the signal from the previous stage (adding delay) or bypass it by injecting the input signal directly. Each cell's control bit determines which path is active.

The cells are chained in series. The head of the chain is initialized to a static value, so it cannot pass a valid signal on its own. The first cell whose control bit enables bypass injects the input clock into the chain. From that point onward, cells with bypass disabled propagate the signal through their NAND logic, each adding gate delay. The effective delay scales with how many propagation stages sit between the injection point and the output. This implementation uses 4 cells with a 4-bit control word. Synthesis annotations prevent the tool from optimizing away the intentional delay structure.

| Pros | Cons |
|----------|----------|
| Wider delay range per stage than inverter-based designs, since each NAND cell introduces more gate delay than a single inverter. | Non-linear delay steps; the delay contribution of each cell depends on its position and the state of other cells. |
| Synthesis-robust: annotations preserve the intentional delay structure through the tool flow. | More complex per-cell logic (three NAND gates and an inverter per stage). Higher area and power per stage than inverter-based designs. |
| Wider control word gives more delay range than the inverter-based DCDLs. | Requires priority-based control encoding. The first enabled bypass from the head of the chain determines the injection point; arbitrary bit patterns may produce unexpected outputs or stuck-at-zero. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py nand_dcdl



> **TODO: Add simulation results / additional waveform figures for NAND-based DCDL**

> **TODO: Add data analysis (delay range, per-step nonlinearity)**

---

### Implementation 5: Vernier Crossover DCDL

> **TODO: Add draw.io diagram (dual chain: slow buf_1 chain and fast buf_4 chain with crossover muxes at each stage controlled by Q[i], output from fast_net[STAGES])**

The most sophisticated DCDL architecture here, targeting fine delay resolution through interpolation between two parallel buffer chains with different per-stage delays. A "slow" chain is built from small, high-delay buffers and a "fast" chain from large, low-delay buffers. At each stage, a mux can cross the signal from the slow chain into the fast chain. The input enters the slow chain; the fast chain is initialized to a static value.

The control word determines where the crossover happens. When a control bit is asserted at a given stage, the slow-chain signal crosses into the fast chain and propagates through the remaining fast-chain stages to the output. More slow stages before the crossover means more total delay. The achievable delay step is the difference between one slow-buffer delay and one fast-buffer delay, which can be much smaller than either buffer's absolute delay. This implementation uses 16 stages with `sky130_fd_sc_hd__buf_1` (slow) and `sky130_fd_sc_hd__buf_4` (fast) standard cells.

| Pros | Cons |
|----------|----------|
| Very fine delay resolution; the delay step is the difference between the slow and fast buffer delays, enabling sub-gate-delay granularity. Best resolution of all DCDL variants. | Largest area and power of all five implementations. Two full buffer chains plus a mux per stage. Highest static and dynamic power. |
| Many stages with per-stage control gives high granularity. | Delay linearity depends on matching between the slow and fast cell delays across PVT corners. |
| Uses sky130 standard cells directly. Ready for physical implementation with no custom cell design. | The fast chain is initialized to a static value, so the output is invalid until at least one crossover mux is enabled. Both chains are always active, wasting power on the unused path. |


In [ ]:
!python3 CAC_2026/scripts/flow_cli.py vernier_dcdl



> **TODO: Add simulation results / waveform figures for vernier DCDL**

> **TODO: Add data analysis (delay resolution measurement, linearity across PVT corners)**

## Ring Oscillator
---

The ring oscillator is not part of the DLL feedback loop itself. It is a tunable on-chip clock source used for testing and characterization. It generates a free-running clock by feeding the output of an odd-length inverter chain back to its input, creating a self-sustaining oscillation. The oscillation frequency is set by the total propagation delay around the loop: `f = 1 / (2 * N * t_inv)`, where `N` is the number of inverter stages and `t_inv` is the delay per inverter.

---

### Implementation: Selectable-Length Inverter Ring

> **TODO: Add draw.io diagram (9-inverter chain with feedback from taps n2/n4/n6/n8 through 4:1 mux controlled by sel[1:0] back to clk_out)**

A chain of inverters is driven from the output back through itself. A select input chooses which tap along the chain feeds back to the output, effectively changing the number of inverters in the oscillation loop. Shorter loops have less total delay and oscillate faster. Each selectable configuration uses an odd number of inversions, which is required for sustained oscillation.

This implementation uses 9 inverters with a 2-bit select, giving four frequency options: 3, 5, 7, or 9 inverters in the loop.

| Pros | Cons |
|----------|----------|
| Simple, compact design. Just inverters and a mux. Minimal area and power for an on-chip clock source. | Frequency is highly sensitive to PVT variation since it depends entirely on inverter delay. Performance is unpredictable across corners. |
| Provides multiple discrete frequency options from a single oscillator. No external clock input required. | The combinational feedback style may cause simulation issues in some tools (zero-delay oscillation in RTL sim without gate-level timing). |
| Useful for standalone testing of the DCDL or full DLL loop. | Frequency selection is coarse. Only a few options with large gaps between them. No fine-grained frequency tuning. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py ring_oscillator



> **TODO: Add simulation results / waveform figures for ring oscillator at each sel value**

> **TODO: Add data analysis (measured frequency vs. sel setting, PVT sensitivity if available)**

## Practical Examples
---